# Nexus: controlled temporal-model experiment
Individual signs only; not sentence translation. This notebook has not been executed in Colab. Use a private runtime and never publish the feature bundle or trained weights. Free GPU access is optional and not guaranteed. See `docs/COLAB.md`.

In [ ]:
import sys, subprocess, shutil, os
from pathlib import Path
print(sys.version)
print("Free disk GiB:", shutil.disk_usage("/content").free / 2**30)
assert sys.version_info[:2] in [(3, 11), (3, 12)], "Use Python 3.11/3.12 for the pinned experiment"
assert shutil.disk_usage("/content").free > 5 * 2**30
REVISION = "main"  # For exact reproduction replace with the recorded commit SHA.
subprocess.run(["git", "clone", "https://github.com/DevVisoulia25/Nexus.git", "/content/Nexus"], check=True)
os.chdir("/content/Nexus")
subprocess.run(["git", "checkout", REVISION], check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"]).decode())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "scikit-learn==1.6.1", "torch==2.5.1"], check=True)
# If Colab asks for a runtime restart, restart, then restore os.chdir("/content/Nexus").

Upload only your private `colab_features.zip`, generated locally with `python -m nexus.cloud_bundle`. This contains cached landmarks, the train/dev manifest and a baseline artifact; no webcam video or final test data.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1 and "colab_features.zip" in uploaded
subprocess.run([sys.executable, "-m", "nexus.cloud_bundle", "--import-file", "colab_features.zip"], check=True)
del uploaded
Path("colab_features.zip").unlink()

In [ ]:
import torch
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE, "Torch:", torch.__version__)
if DEVICE == "cuda": print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "nexus.window_experiment"], check=True)
subprocess.run([sys.executable, "-m", "nexus.temporal", "--epochs", "60", "--device", DEVICE], check=True)

In [ ]:
import json, zipfile
report = Path("artifacts/recognition/temporal_experiment.json")
print(report.read_text())
Path("artifacts/recognition/notebook_environment.txt").write_bytes(subprocess.check_output([sys.executable, "-m", "pip", "freeze"]))
Path("artifacts/recognition/notebook_commit.txt").write_bytes(subprocess.check_output(["git", "rev-parse", "HEAD"]))
with zipfile.ZipFile("private_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [report, Path("artifacts/recognition/window_experiment.json"), Path("artifacts/recognition/notebook_environment.txt"), Path("artifacts/recognition/notebook_commit.txt"), Path("models/signs-temporal-candidate.npz")]:
        z.write(p, p.as_posix())
files.download("private_results.zip")

Do not promote the model based on forced-choice accuracy alone. Compare accepted correct signs and false activations in replay, then measure local webcam behavior. The same development examples calibrated rejection, so these results are not independent validation. Final test stays untouched.